# Using the `analogues` module

First, make sure your environment (here assumed to be named `.venv`) is up to date by running:
```bash
source .venv/bin/activate
pip install .
```

The setup below finds the repo root automatically, imports the explicit classes used in this notebook, resolves the TNG outputs path without relying on a colleague-specific hardcoded location, and loads the analogue selection and filter settings from `config/config.json`. You can override the automatic path discovery with `TNG_BASE_PATH` or `ILLUSTRIS_BASE_PATH`.


In [1]:
import os
import sys
from pathlib import Path

import pandas as pd
from IPython.display import display

cwd = Path.cwd().resolve()
repo_root = next((path for path in [cwd, *cwd.parents] if (path / "analogues").is_dir()), None)
if repo_root is None:
    raise FileNotFoundError("Could not find the repo root containing the `analogues` package.")

if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

from analogues import AnalogueSample, FilterConfig, SelectionConfig
from config.loader import load_config

print("Repo root:", repo_root)


Repo root: /home/tp/Documents/uni/dsp


This gives access to the main public classes used here. The most straightforward way to produce the analogue sample is to use the `AnalogueSample` class. It expects a `base_path` pointing to the TNG outputs directory, a `selection_config` controlling the initial subhalo sample, and a `filter_config` controlling the pair-level cuts. The next cell resolves `base_path` automatically, loads the shared config from `config/config.json`, and then builds the sample.


In [2]:
def resolve_base_path(repo_root: Path) -> str:
    raw_candidates = [
        os.environ.get("TNG_BASE_PATH"),
        os.environ.get("ILLUSTRIS_BASE_PATH"),
        str(repo_root / "tng300" / "outputs"),
    ]

    for raw in raw_candidates:
        if not raw:
            continue
        base = Path(raw).expanduser()
        for candidate in (base, base / "tng300" / "outputs"):
            if (candidate / "groups_099").exists():
                return str(candidate.resolve())

    raise FileNotFoundError(
        "Could not find a valid TNG outputs directory. Set TNG_BASE_PATH or ILLUSTRIS_BASE_PATH "
        "to the outputs directory, or to a root that contains tng300/outputs."
    )


config_loader = load_config()
analysis_config = config_loader.get_config()

base_path = resolve_base_path(repo_root)
snap = 99
print("Using base_path:", base_path)

selection_config = SelectionConfig(**analysis_config.selection)
filter_config = FilterConfig(**analysis_config.filter)

print("Selection config loaded from config/config.json:")
display(pd.DataFrame([analysis_config.selection]))
print("Filter config loaded from config/config.json:")
display(pd.DataFrame([analysis_config.filter]))
print("Active features in shared config:", list(analysis_config.features))
print("Active target in shared config:", analysis_config.target_name)

analogue_sample = AnalogueSample(
    base_path=base_path,
    selection_config=selection_config,
    filter_config=filter_config,
    verbose=True,
    snap=snap,
)


Using base_path: /home/tp/Documents/uni/dsp/tng300/outputs
Selection config loaded from config/config.json:


,m_stellar_min,m_stellar_max,r_min,r_max,blue_threshold_gr
0,2.000000e+10,5.000000e+11,500.0,1000.0,0.65


Filter config loaded from config/config.json:


,v_tot_min,v_tot_max,vt_min,vt_max,vr_min,vr_max,density_radius,intruder_factor,third_massive_factor
0,0,600,0,300,-400,0,2000,0.5,1.5


Active features in shared config: ['r_kpc', 'v_r', 'v_t']
Active target in shared config: larger_halo_m200c_log10

[DIAGNOSTIC] Simulation Header Constants
╒════════════╤═══════════╤═══════════════╕
│ Parameter  │ Value     │ Units         │
├────────────┼───────────┼───────────────┤
│ Hubble (h) │ 0.6774    │ dimensionless │
├────────────┼───────────┼───────────────┤
│ Box Side   │ 302627.69 │ kpc           │
╘════════════╧═══════════╧═══════════════╛

[DIAGNOSTIC] Sample Selection
╒═════════════════════════════════════════╤══════════╤═════════════════╕
│ Selection Step                          │    Count │ Yield           │
╞═════════════════════════════════════════╪══════════╪═════════════════╡
│ Total Subhalos in Catalog               │ 14485709 │ 100.0%          │
├─────────────────────────────────────────┼──────────┼─────────────────┤
│ Passed Quality Flags (SubhaloFlag==1)   │  1902558 │ 13.1%           │
├─────────────────────────────────────────┼──────────┼─────────────────┤
│

Finally the analogue pairs can be accessed via the `pairs` attribute:


In [3]:
pairs = analogue_sample.pairs
print(pairs)

PairSet(i=array([ 9426,  3877,  6967, ..., 18686, 14316, 14723], shape=(1761,)), j=array([ 9427,  3878,  6968, ..., 37564, 14317, 40978], shape=(1761,)), have_same_host=array([ True,  True,  True, ..., False,  True, False], shape=(1761,)), is_tidally_dominant=array([False, False, False, ..., False, False, False], shape=(1761,)), is_blue_blue=array([ True, False, False, ..., False, False, False], shape=(1761,)), is_red_red=array([False,  True,  True, ..., False, False, False], shape=(1761,)), is_blue_red=array([False, False, False, ...,  True,  True,  True], shape=(1761,)), separation=array([684.23224, 516.9462 , 798.1973 , ..., 852.0389 , 716.65717,
       941.0283 ], shape=(1761,), dtype=float32), vr=array([-203.14726 , -221.99469 , -265.46622 , ...,  -67.23894 ,
       -191.03812 ,  -95.240204], shape=(1761,), dtype=float32), vt=array([239.42361,  68.69453, 268.23666, ..., 220.71962,  74.58484,
        97.1697 ], shape=(1761,), dtype=float32), force_ratio=array([ 1.65695812, 13.62020